# Stag Hunt sweep on Colab



## 1. Setup: paths, Drive, helper

In [ ]:
import os, subprocess, time, signal
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/staghunt'   # zip in, results out (survives disconnects)
CODE  = '/content/stag-hunt-with-RL'        # code on the fast local disk (wiped on disconnect)
os.makedirs(DRIVE, exist_ok=True)

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable='/bin/bash')
    out = '\n'.join(l for l in (r.stdout + r.stderr).splitlines()
                    if 'pygame' not in l.lower() and 'hello from' not in l.lower())
    print(out)

def sweep_procs():
    # (pid, kind) for our own processes only: 'train' = python train.py ..., 'launcher' = bash run_all.sh
    out = []
    for line in subprocess.run(['ps', '-eo', 'pid,args'], capture_output=True, text=True).stdout.splitlines()[1:]:
        pid, *tok = line.split()
        if tok and 'python' in tok[0] and any(t.endswith('train.py') for t in tok[1:4]):
            out.append((int(pid), 'train'))
        elif len(tok) >= 2 and tok[0].endswith('bash') and tok[1].endswith('run_all.sh'):
            out.append((int(pid), 'launcher'))
        elif tok and tok[0].endswith('xargs') and 'run_one' in line:
            out.append((int(pid), 'xargs'))
    return out

def n_training():
    p = sweep_procs()
    return sum(k == 'train' for _, k in p), any(k == 'launcher' for _, k in p)

sh('echo cores: $(nproc); free -h | head -2')
print('Drive folder:', os.listdir(DRIVE))

## 2. Code and environment


In [ ]:
if not os.path.exists(f'{CODE}/train.py'):
    sh(f'''
    mkdir -p {CODE} && unzip -q -o {DRIVE}/stag-code.zip -d {CODE}
    if [ ! -f {CODE}/train.py ]; then sub=$(dirname $(find {CODE} -name train.py | head -1)); mv "$sub"/* {CODE}/; fi
    ''')
else:
    print('code already unpacked')

if not os.path.exists('/content/Gymnasium-Stag-Hunt'):
    sh('cd /content && git clone -q https://github.com/giorgiofranceschelli/Gymnasium-Stag-Hunt.git')
sh('pip install -q -e /content/Gymnasium-Stag-Hunt')

# outputs live on Drive; the code folder links to them
for d in ['runs', 'logs', 'done', 'results']:
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
    p = f'{CODE}/{d}'
    if not os.path.islink(p):
        sh(f'rm -rf {p} && ln -s {DRIVE}/{d} {p}')

sh(f'cd {CODE} && python -c "import torch, gymnasium_stag_hunt, train; print(\'OK\', torch.__version__)"')

## 3. Smoke test (optional, ~1 min)
Expect two `step ...` lines and a `final greedy evaluation` block. Skip it if the sweep is already running.

In [ ]:
sh(f'cd {CODE} && python train.py --total-steps 2000 --eval-every 1000 --eval-episodes 5 '
   f'--final-eval-episodes 5 --out /content/test_runs; rm -rf /content/test_runs')

## 4. Run list
Built once and kept on Drive (`commands.txt`), so it never changes under a running sweep. Re-running this cell only appends runs that are not in the list yet.

All runs of the project, in the order they were launched (same list as `experiments/run_list.txt` in the repo):
1. **Default environment, 48 runs**: hysteretic β=0.2, NoisyNet per step, NoisyNet per episode, ε-start 0.1 control × shared/separate × α∈{0, 0.5} × 3 seeds.
2. **Plain DQN baseline, 12 runs**: shared/separate × α∈{0, 0.5} × 3 seeds.
3. **Environment sweep, 141 runs** (separate networks, α=0, 5 seeds; plain DQN, hysteretic, NoisyNet per episode): seeds 3–4 at default settings, mauling penalty −1/−2/−10/−20, random stag, grid 7×7 and 9×9, forage reward 2 and 3.

In [ ]:
import sys, shlex
sys.path.insert(0, CODE)
sys.path.insert(0, '/content/Gymnasium-Stag-Hunt')   # the env package, installed by cell 2
import train
assert hasattr(train, 'env_label'), 'old train.py on Colab: re-upload stag-code.zip and restart the runtime'

def sweep(opts, extra=''):
    out = subprocess.run(f'cd {CODE} && python sweep.py {opts} --print-commands -- {extra}',
                         shell=True, capture_output=True, text=True).stdout
    return [l for l in out.splitlines() if 'train.py' in l]

lines = []
# 1. default environment: learning-rule variants (48 runs)
for mode in ['', '--mode separate']:
    for v in ['--hyst-beta 0.2', '--noisy', '--noisy --noise-resample episode', '--eps-start 0.1']:
        lines += sweep('--alphas 0 0.5 --mauls -5', f'{mode} {v}')
# 2. plain DQN baseline (12 runs)
for mode in ['', '--mode separate']:
    lines += sweep('--alphas 0 0.5 --mauls -5', mode)
# 3. environment sweep (141 runs)
METHODS = ['', '--hyst-beta 0.2', '--noisy --noise-resample episode']   # plain DQN, hysteretic, NoisyNet per episode
SEEDS = '10000 10001 10002 10003 10004'
CONDITIONS = [('--mauls -5', ''),                      # default env (adds seeds 3 and 4)
              ('--mauls -1 -2 -10 -20', ''),           # mauling penalty
              ('--mauls -5', '--no-stag-follows'),     # stag moves randomly
              ('--mauls -5', '--grid-size 7 7'),
              ('--mauls -5', '--grid-size 9 9'),
              ('--mauls -5', '--forage-reward 2'),
              ('--mauls -5', '--forage-reward 3')]
for opts, env_flags in CONDITIONS:
    for m in METHODS:
        lines += sweep(f'--modes separate --alphas 0 {opts} --seeds {SEEDS}', f'{m} {env_flags}')

# append only runs whose name is not in the list yet (the same run can be generated twice)
CMDS = f'{DRIVE}/commands.txt'
existing = open(CMDS).read().splitlines() if os.path.exists(CMDS) else []
name_of = lambda l: train.parse_args(shlex.split(l)[2:]).name
known = {name_of(l) for l in existing if 'train.py' in l}
new = []
for l in lines:
    if name_of(l) not in known:
        known.add(name_of(l)); new.append(l)
if new:
    open(CMDS, 'a').write('\n'.join(new) + '\n')
total = len(open(CMDS).read().splitlines())
print(f'{len(new)} runs added | total in list: {total}')   # 201 runs in total


## 5. Launch / resume
One run per core. Finished runs (marker in `done/`) are skipped; unfinished ones restart from zero.
If a sweep is already running, this cell does nothing.

In [ ]:
open('/content/run_all.sh', 'w').write(f'''#!/bin/bash
cd {CODE}
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 CUDA_VISIBLE_DEVICES=   # CPU only, one thread per run
run_one() {{
  i=$1
  [ -f done/$i ] && exit 0
  cmd=$(sed -n "${{i}}p" {DRIVE}/commands.txt)
  echo "$cmd" > logs/run_$i.log
  bash -c "$cmd" >> logs/run_$i.log 2>&1 && touch done/$i
}}
export -f run_one
seq 1 $(wc -l < {DRIVE}/commands.txt) | xargs -P {os.cpu_count()} -I{{}} bash -c 'run_one {{}}'
echo ALL FINISHED
''')

n, launcher = n_training()
if launcher or n > 0:
    print(f'Sweep already running ({n} training processes) - not launching again.')
else:
    log = open(f'{DRIVE}/sweep.log', 'a')
    subprocess.Popen(['bash', '/content/run_all.sh'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(15)
    print('launched - training processes now:', n_training()[0])

## 6. Progress (re-run whenever you like)

In [ ]:
n, launcher = n_training()
total = len(open(f'{DRIVE}/commands.txt').read().splitlines())
print(f'finished: {len(os.listdir(f"{DRIVE}/done"))} / {total} | running: {n} | launcher alive: {launcher}')
sh(f'''cd {DRIVE}/logs; for f in $(ls -t *.log 2>/dev/null | head -4); do
      echo "== $f: $(grep '^step' $f | tail -1 | cut -c1-120)"; done
      tail -1 {DRIVE}/sweep.log 2>/dev/null''')

## 7. Results (when finished = total)
Prints the per-configuration table and writes `results/summary.csv`. All runs are on Drive in `MyDrive/staghunt/runs/`.
When done: Runtime → Disconnect and delete runtime, so it stops using compute units.

In [ ]:
sh(f'cd {CODE} && python aggregate.py')

##  Emergency only: stop everything
Kills all training. Finished runs stay; unfinished ones restart from zero next time you run cell 5.

In [ ]:
STOP = False   # set to True and run to stop the sweep
if STOP:
    for pid, kind in sweep_procs():
        try: os.kill(pid, signal.SIGKILL)
        except ProcessLookupError: pass
    time.sleep(3)
    print('training processes left:', n_training()[0])